# tool_eval — 에이전트 도구 성능 평가 (Jupyter)

`도구-eval-가이드.md` (Anthropic *Writing effective tools for agents*) 의 방법론을
**미니 CC 의 목 저장소(orderhub)** 에 적용한, 셀 단위로 돌아가는 eval.
평가 대상은 코드 탐색 도구 3종 — `read_file` / `search_code` / `list_files`.
이 도구들의 **설명(description)·스키마** 가 측정·개선 대상이다.

| 노트북 구획 | 가이드 단계 |
|---|---|
| 평가 대상 도구 (SUT) | 도구 정의 — description 을 고쳐가며 성능 변화 관찰 |
| 1. 평가 과제 | **Stage 1** 현실적·다단계 과제 6개(train 4 / heldout 2) + 느슨한 검증기 |
| 2. 실행 (agentic loop) | **Stage 2** 과제당 while 루프 하나 (OpenAI Responses) |
| 3. 실행 & 분석 | **Stage 3** 정확도 + 호출수·토큰·오류·시간 + 원본 로그 읽기 |
| 4. 개선 루프 | **Stage 4** held-out 과적합 점검 + tool_feedback → 설명 수정 |

> 의존성: orderhub 목 FS 는 `notebooks/claude_code/cc_mock_fs.py` 를 재사용한다(설정 셀이 자동 경로 탐색). LLM 은 레포 루트 `.env` 의 `OPENAI_API_KEY`.

## 0. 셋업 — 목 FS + OpenAI 클라이언트

In [1]:
import os, sys, json, time, re, pathlib
from dataclasses import dataclass, field
from typing import Callable

# orderhub 목 파일시스템은 미니 CC(claude_code)의 파일을 재사용한다 — 상위 경로에서 자동 탐색
def find_up(rel):
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        p = base / rel
        if p.exists():
            return p
    return None

cc = find_up("claude_code/cc_mock_fs.py") or find_up("notebooks/claude_code/cc_mock_fs.py")
sys.path.insert(0, str(cc.parent))
from cc_mock_fs import FS
print(f"목 파일시스템: {len(FS)}개 파일 (orderhub)")

# OpenAI 클라이언트 (레포 루트 .env 의 OPENAI_API_KEY)
try:
    from dotenv import load_dotenv
    env = find_up(".env")
    if env:
        load_dotenv(env)
except Exception:
    pass
from openai import OpenAI
MODEL = "gpt-5-nano"  # 레포 관례
client = OpenAI() if os.getenv("OPENAI_API_KEY") else None
print("OpenAI:", "준비됨 (실행 셀 사용 가능)" if client else "키 없음 — 실행 셀은 건너뜀")

목 파일시스템: 40개 파일 (orderhub)
OpenAI: 준비됨 (실행 셀 사용 가능)


## 평가 대상 도구 (SUT)

일부러 완벽하지 않은 지점을 남겼다 — `search_code` 는 **대소문자 구분**. Stage 4 에서 이걸 로그로 잡아 고친다.

In [2]:
SEARCH_CAP = 40  # grep 결과 상한 (rightsizing/중복호출 관측용)

def _numbered(content):
    return "\n".join(f"{i:>4}  {ln}" for i, ln in enumerate(content.splitlines(), 1))

def _glob_to_rx(glob):
    esc = re.escape(glob or "*").replace(r"\*", ".*").replace(r"\?", ".")
    return re.compile(esc)

def read_file(path):
    if path not in FS:
        alt = next((p for p in FS if p.endswith(path.lstrip("/"))), None)
        hint = f" 혹시 이 경로? {alt}" if alt else " list_files 로 경로를 먼저 확인하라."
        return f"<error>파일 없음: {path}.{hint}</error>"
    return _numbered(FS[path])

def search_code(pattern, path_glob="*"):
    try:
        rx = re.compile(pattern)  # 대소문자 구분 (의도된 거칢 — Stage 4 참고)
    except re.error as e:
        return f"<error>정규식 오류: {e}</error>"
    globrx = _glob_to_rx(path_glob)
    hits = []
    for p in sorted(FS):
        if not globrx.search(p):
            continue
        for i, ln in enumerate(FS[p].splitlines(), 1):
            if rx.search(ln):
                hits.append(f"{p}:{i}: {ln.strip()[:120]}")
    body = "\n".join(hits[:SEARCH_CAP]) or "(일치 없음)"
    if len(hits) > SEARCH_CAP:
        body += f"\n… (+{len(hits) - SEARCH_CAP}건 더 — pattern 이나 path_glob 을 좁히세요)"
    return body

def list_files(path_glob="*"):
    globrx = _glob_to_rx(path_glob)
    out = [p for p in sorted(FS) if globrx.search(p)]
    return "\n".join(out) or "(일치 없음)"

# ── Responses API 툴 스키마 (flat: type/name/description/parameters) ──
TOOLS = [
    {"type": "function", "name": "read_file",
     "description": ("지정한 파일 전체 내용을 1-기반 줄 번호와 함께 반환. path 는 '/project/...' 절대경로. "
                     "경로가 불확실하면 먼저 list_files/search_code 로 확인하라."),
     "parameters": {"type": "object",
        "properties": {"path": {"type": "string", "description": "읽을 파일 절대경로. 예: /project/src/app/services/order_service.py"}},
        "required": ["path"]}},
    {"type": "function", "name": "search_code",
     "description": ("저장소 전체를 Python 정규식으로 검색. 결과는 'path:line: 내용' 형식, 최대 40건. "
                     "함수명·티켓번호(ORDER-482)·TODO/FIXME 토큰 찾기에."),
     "parameters": {"type": "object",
        "properties": {"pattern": {"type": "string", "description": r"Python 정규식. 예: 'def calc_total' 또는 'ORDER-\d+'"},
                       "path_glob": {"type": "string", "description": "경로 필터(단순 glob, 부분일치). 예: '*.py','services'. 기본 '*'"}},
        "required": ["pattern"]}},
    {"type": "function", "name": "list_files",
     "description": "경로 glob 에 맞는 파일 목록 반환(부분일치). 구조 파악·경로 확인용. 예: '*.py','routers'.",
     "parameters": {"type": "object",
        "properties": {"path_glob": {"type": "string", "description": "경로 glob. 예: 'src/app/services/*.py'. 기본 '*'"}}}},
]

_IMPL = {"read_file": read_file, "search_code": search_code, "list_files": list_files}

def run_tool(name, args):
    fn = _IMPL.get(name)
    if fn is None:
        return f"<error>알 수 없는 도구: {name}</error>"
    try:
        return fn(**args)
    except TypeError as e:
        return f"<error>인자 오류({name}): {e}</error>"

# 스모크 (API 불필요)
print(search_code(r"ORDER-\d+").splitlines()[0])

/project/README.md:25: - ORDER-482: 쿠폰 할인이 배송비에도 적용되는 버그 (order_service 참고)


## 1. 평가 과제 (Stage 1)

답·방법이 프롬프트에 없다. 에이전트가 도구를 조합해 스스로 경로를 찾아야 한다.
검증기는 '개념별 동의어 집합'이 모두 등장하는지만 보는 느슨한 매칭(형식 차이로 오답 처리 X).

In [3]:
@dataclass
class Task:
    id: str
    split: str            # "train" | "heldout"
    prompt: str
    verify: Callable
    expected_tools: list = field(default_factory=list)

def _kw(*groups):
    # groups = (개념라벨, [동의어...]) 들. 각 그룹에서 하나라도 등장하면 충족, 모두 충족 시 정답.
    def verify(text):
        low = (text or "").lower()
        missing = [label for label, syns in groups if not any(s.lower() in low for s in syns)]
        return (False, "누락 개념: " + ", ".join(missing)) if missing else (True, "필수 개념 모두 언급")
    return verify

EVAL_TASKS = [
    Task("T1-coupon-shipping", "train",
         "한 고객이 12,000원짜리 상품 하나를 쿠폰 WELCOME5로 주문했는데 청구 금액이 예상과 다르다고 한다. "
         "코드에서 원인이 되는 함수를 위치와 함께 찾고, 정확히 무엇이 잘못됐는지 설명하라.",
         _kw(("함수/위치", ["calc_total", "order_service"]),
             ("버그 근거", ["order-482", "배송비", "shipping"])),
         ["search_code", "read_file"]),
    Task("T2-duplicate-coupon", "train",
         "같은 쿠폰을 두 번 이상 쓸 수 있는 것 같다는 제보가 들어왔다. 코드에서 중복 사용을 막지 않는 지점을 찾고, "
         "팀 할 일 목록에 이 문제 관련 항목이 있는지도 확인해 정리하라.",
         _kw(("위치", ["create", "order_service"]), ("티켓", ["order-517"]), ("할일 근거", ["todo", "할 일", "이력"])),
         ["search_code", "read_file"]),
    Task("T3-password-hash", "train",
         "비밀번호 해시 방식이 보안상 약하다는 지적이 있었다. 현재 어떤 방식을 쓰는지, 개선 티켓 번호는 무엇인지, "
         "무엇으로 바꾸기로 돼 있는지 근거와 함께 정리하라.",
         _kw(("현재 방식", ["sha256"]), ("티켓", ["auth-201"]), ("개선 목표", ["bcrypt"])),
         ["search_code", "read_file"]),
    Task("T4-payment-timeout", "train",
         "결제가 30초를 넘겨 실패한 흔적이 로그에 있는지 확인하고, 있다면 어떤 요청이었는지와 재시도 횟수가 "
         "코드상 몇 회로 설정돼 있는지 근거와 함께 답하라.",
         _kw(("로그 근거", ["timeout", "30s", "30초"]), ("요청 대상", ["charge", "결제", "payment", "pay.example"]),
             ("재시도", ["retry_limit", "3회", "retry", "재시도"])),
         ["search_code", "read_file"]),
    Task("T5-deprecated-payment", "heldout",
         "폐기 예정(deprecated)이라고 표시된 결제 함수가 있다고 한다. 그 함수명과 폐기 사유/시점, 대신 써야 할 함수, "
         "관련 티켓 번호를 정리하라.",
         _kw(("폐기 함수", ["charge_legacy"]), ("대체 함수", ["request_payment"]), ("티켓", ["pay-77"])),
         ["search_code", "read_file"]),
    Task("T6-pagination-gap", "heldout",
         "users 목록 API에는 아직 적용되지 않았지만 다른 곳에는 이미 적용된 개선이 하나 있다. 그 개선이 무엇인지, "
         "users 쪽 현황과 이미 적용된 곳을 근거와 함께 답하라.",
         _kw(("개선 내용", ["페이지네이션", "pagination", "paginate"]), ("users 현황", ["users", "list_users", "사용자 목록"]),
             ("적용된 곳", ["상품", "product", "검색", "search"])),
         ["search_code", "read_file"]),
]
print(f"과제 {len(EVAL_TASKS)}개 (train {sum(t.split=='train' for t in EVAL_TASKS)} / heldout {sum(t.split=='heldout' for t in EVAL_TASKS)})")

과제 6개 (train 4 / heldout 2)


## 2. 평가 실행 — agentic loop (Stage 2)

시스템 프롬프트가 도구 호출 앞 `<plan>`, 끝에 `<answer>`+`<tool_feedback>` 를 강제해 CoT 를 유발한다.
과제 하나당 while 루프 하나, 각 에이전트에겐 단일 과제 프롬프트 + 도구들만 준다.

In [4]:
SYSTEM_PROMPT = (
    "너는 orderhub 코드베이스를 조사하는 에이전트다. 주어진 도구만으로 저장소를 탐색해 사용자 질문에 답하라.\n"
    "규칙:\n"
    "1) 도구를 호출하기 전에 항상 <plan>...</plan> 블록으로 무엇을 왜 찾을지 한두 문장 적어라.\n"
    "2) 파일 경로·줄·티켓번호 같은 근거를 실제 도구 결과에서 확인한 뒤에만 결론을 내려라. 추측 금지.\n"
    "3) 마지막엔 <answer>...</answer> 로 최종 답을, 이어서 <tool_feedback>...</tool_feedback> 로 "
    "도구 설명/스키마에서 헷갈렸거나 개선하면 좋을 점을 한두 줄 남겨라.\n"
)

def run_task(client, task, model=None, max_turns=12):
    model = model or MODEL
    input_list = [{"role": "user", "content": task.prompt}]
    transcript, used_tools = [], []
    n_calls = n_errors = in_tok = out_tok = 0
    final_text = ""
    t0 = time.time()
    for _ in range(max_turns):
        resp = client.responses.create(model=model, instructions=SYSTEM_PROMPT,
                                       input=input_list, tools=TOOLS, parallel_tool_calls=True)
        if resp.usage:
            in_tok += resp.usage.input_tokens
            out_tok += resp.usage.output_tokens
        input_list += resp.output
        if resp.output_text.strip():
            transcript.append(("assistant", resp.output_text.strip()))
        calls = [it for it in resp.output if it.type == "function_call"]
        if not calls:
            final_text = resp.output_text
            break
        for c in calls:
            try:
                args = json.loads(c.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            result = run_tool(c.name, args)
            n_calls += 1
            used_tools.append(c.name)
            if result.startswith("<error>"):
                n_errors += 1
            transcript.append(("tool_call", f"{c.name}({json.dumps(args, ensure_ascii=False)})"))
            transcript.append(("tool_result", result))
            input_list.append({"type": "function_call_output", "call_id": c.call_id, "output": result})
    else:
        final_text = final_text or "(max_turns 도달)"
    passed, reason = task.verify(final_text)
    return {"task_id": task.id, "split": task.split, "passed": passed, "reason": reason,
            "tool_calls": n_calls, "tool_errors": n_errors, "input_tokens": in_tok, "output_tokens": out_tok,
            "duration_s": round(time.time() - t0, 2),
            "missing_expected_tools": [t for t in task.expected_tools if t not in used_tools],
            "final_text": final_text, "transcript": transcript}

## 3. 실행 & 분석 (Stage 3)

먼저 과제 하나만 돌려 **원본 호출 로그를 눈으로 읽는다.** 정확도 숫자만으로는 안 잡히는
헛도는 호출·잘못된 파라미터를 여기서 본다.

In [5]:
# 과제 하나 실행 + 로그 읽기. client 없으면 건너뜀.
if client:
    r = run_task(client, EVAL_TASKS[0])
    print(("✓ PASS" if r["passed"] else "✗ FAIL"), "—", r["reason"])
    print(f"도구호출 {r['tool_calls']} · 오류 {r['tool_errors']} · "
          f"토큰 {r['input_tokens']+r['output_tokens']:,} · {r['duration_s']}s")
    print("\n── 최종 답 (앞부분) ──")
    print(r["final_text"][:1200])
    print("\n── 원본 호출 로그 ──")
    for role, content in r["transcript"]:
        print(f"[{role}] {content[:280]}")
else:
    print("OPENAI_API_KEY 없음 — 이 셀 건너뜀")

✓ PASS — 필수 개념 모두 언급
도구호출 8 · 오류 0 · 토큰 20,742 · 56.53s

── 최종 답 (앞부분) ──
<plan>쿠폰 WELCOME5의 적용 로직이 어디서 어떻게 계산되는지 확인하고, 12,000원 상품에 대한 청구 금액이 왜 기대와 다르게 나오는지 관련 함수 위치와 잘못된 동작을 근거와 함께 설명하겠다.</plan>

<answer>
요지
- 문제의 원인 함수: 주문 총액 계산 로직인 OrderService.calc_total(...)가 쿠폰을 총합(total)에 바로 적용하도록 구현되어 있어, 쿠폰이 배송비를 포함한 총액에도 차감되도록 동작한다.
- 근거 위치
  - 쿠폰 코드 매핑 및 조회 로직: /project/src/app/repositories/order_repo.py의 find_coupon
    - 코드 내에서 WELCOME5가 5_000으로 매핑되어 있다.
    - 코드 예: promotions = {"WELCOME5": 5_000, "VIP10": 10_000}
  - 쿠폰 적용 로직: /project/src/app/services/order_service.py의 calc_total
    - 총액 계산 후 쿠폰이 있으면 total = clamp(total - coupon.amount, 0, total)로 차감을 수행한다.
    - 해당 부분에 코멘트(FIXME)로 “할인이 배송비 포함 총액에 걸린다. 정책상 쿠폰은 상품 금액(subtotal)에만 적용해야 한다.”라고 명시되어 있다.
    - 코드 예:
      - subtotal 계산
      - shipping 계산
      - if coupon: total = clamp(total - coupon.amount, 0, total)
  - 의도한 정책과의 차이: 주석의 FIXME에 따라 쿠폰은 배송비를 포함한 총액이 아닌, 상품 금액(subtotal)에만 적용되어야 한다. 현재 구현은 총액에 쿠폰을 차감하므로 배송비도 할인에 포함될 여지가 있다.
 

### 전체 실행 + 집계 (선택)

정확도만 보지 말 것 — 호출수·토큰·오류·시간을 함께, train↔heldout 격차로 과적합을 본다.

In [ ]:
def summarize(results):
    for sp in sorted({r["split"] for r in results}):
        rs = [r for r in results if r["split"] == sp]; n = len(rs)
        print(f"[{sp:>7}] 정확도 {sum(x['passed'] for x in rs)/n:5.0%} ({sum(x['passed'] for x in rs)}/{n}) · "
              f"평균 도구호출 {sum(x['tool_calls'] for x in rs)/n:.1f} · 도구오류 {sum(x['tool_errors'] for x in rs)} · "
              f"평균 토큰 {sum(x['input_tokens']+x['output_tokens'] for x in rs)/n:,.0f} · 평균 {sum(x['duration_s'] for x in rs)/n:.1f}s")

if client:
    results = [run_task(client, t) for t in EVAL_TASKS]
    for r in results:
        print(("✓" if r["passed"] else "✗"), r["task_id"], "—", r["reason"])
    print()
    summarize(results)
    outdir = pathlib.Path.cwd() / "results"; outdir.mkdir(exist_ok=True)
    outfile = outdir / f"{time.strftime('%Y%m%d-%H%M%S')}.jsonl"
    outfile.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in results))
    print("\n저장:", outfile)
else:
    print("OPENAI_API_KEY 없음 — 이 셀 건너뜀")

## 4. 개선 루프 (Stage 4)

- **held-out 과적합 점검**: `heldout` split(T5·T6)은 도구 설명을 튜닝하며 보지 않는다. train↔heldout 정확도 격차가 크면 과적합.
- **tool_feedback**: 각 답 끝의 `<tool_feedback>` + 원본 로그를 이어붙여 Claude Code 에 주면 도구를 일괄 개선해 준다.
- **의도된 거칢 실습**: `search_code` 는 대소문자를 구분한다. 에이전트가 `todo` 로 검색해 `TODO` 를 놓치면 로그에 "빈 결과 → 재검색" 이 보인다. 고칠 것은 모델이 아니라 **도구** — description 에 "대소문자 구분" 을 명시하거나 구현을 `re.IGNORECASE` 로 바꾸고 eval 을 다시 돌려 **도구오류/낭비 호출이 줄었는지 숫자로** 확인한다.

> 라이브 실행에서 관측된 예: 에이전트가 존재하지 않는 `apply_coupon|calculate_total` 같은 영어 함수명을 지어내 검색 → 0건 낭비. 정확도(PASS)만 보면 안 잡히고 로그를 읽어야 보이는 신호다. 이게 가이드가 말하는 "2025 편향" 과 같은 종류의 발견이다.